# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd

In [2]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [3]:
# explorar datasets
orders.info()
orders.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [4]:
catalog.info()
catalog.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [5]:
marketing.info()
marketing.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [6]:
#---------------------------------------------DF_ORDERS-------------------
#tengo 300 valores nulos en pais
#los otros datos que venian en minusculas se corrigieron para normalizar los datos
#Se eliminaron 50 datos que no representan gran porcentaje, ya que tanto cantidad y precio unitario estaban ausentes y no se podian encontrar.
orders['pais']=orders['pais'].str.strip().str.title()
print("-----------Datos Pais--------")
print(orders['pais'].unique())
print()
print("-----------pruebas_columnas--------")
print(orders['categoria_producto'].unique())
print()
orders["fecha_hora_pedido"]=pd.to_datetime(orders["fecha_hora_pedido"])
orders = orders.dropna(subset=['cantidad', 'precio_unitario'])
print(orders.info())

-----------Datos Pais--------
['Argentina' 'Mexico' 'Colombia' nan]

-----------pruebas_columnas--------
['Moda' 'Electronica' 'Hogar' nan]

<class 'pandas.core.frame.DataFrame'>
Int64Index: 25050 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25050 non-null  object        
 1   id_usuario          25050 non-null  object        
 2   fecha_hora_pedido   25050 non-null  datetime64[ns]
 3   pais                24750 non-null  object        
 4   dispositivo         25030 non-null  object        
 5   fuente_referencia   25020 non-null  object        
 6   nombre_producto     25020 non-null  object        
 7   categoria_producto  25020 non-null  object        
 8   cantidad            25050 non-null  float64       
 9   precio_unitario     25050 non-null  float64       
 10  monto_descuento     25050 non-null  float64       
 11  monto_total      

In [7]:
#---------------------------------------------DF_CATALOG-------------------
#No hay problemas con los datos

In [8]:
#---------------------------------------------DF_MARKETING-------------------
#Los canales los podemos extraer del id_campaña
marketing["canal"].unique()

array(['organic', 'paid_search', 'social', nan], dtype=object)

In [9]:
canales_validos = ['organic', 'paid_search', 'social']
def extraer_canal(id_campaña):
    for canal in canales_validos:
        if id_campaña.startswith(canal):
            return canal
    return np.nan

mask = marketing['canal'].isna()
marketing.loc[mask, 'canal'] = marketing.loc[mask, 'id_campaña'].apply(extraer_canal)

In [10]:
print(marketing["canal"].unique())
print()
print(marketing["canal"].isna().sum())

['organic' 'paid_search' 'social']

0


In [11]:
marketing["fecha"] = pd.to_datetime(marketing["fecha"])

marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1620 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 63.4+ KB


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [12]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? **El ingreso total es: 52002629**
- ¿Cuál es el costo total? **Costo total es: 43140620.46**
- ¿Cuánto se ha invertido en marketing? **Gasto en Marketing es: 2871843.53**
- gasto por camapaña  
▸ **organic Argentina  :     319196.99**  
▸**organic Colombia   :      323222.51**  
▸**organic Mexico     :      330231.46**  
▸**paid search Argentina :   297321.98**  
▸**paid search Colombia  :   298901.96**  
▸**paid search Mexico    :   326150.26**  
▸**social Argentina      :   331175.63**  
▸**social Colombia       :   313528.95**  
▸**social Mexico         :   332113.79**  
- ¿El negocio es rentable? (calcular profit)  
**Después de cubrir todos los costos y gastos, al negocio le sobró dinero, es decir generó una ganancia de: $5990165.57**
---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? **El ticket promedio por pedido: $2,084.27**
- ¿Cuál es la cantidad promedio de productos por orden? **Cantidad promedio de productos por orden: 7.12u**
- ¿Cuál es el producto más vendido? **El producto más vendido es: Laptop-Gaming-16GB**
- ¿Cuánto se ha gastado en marketing por canal?  
▸ **organic       :      972650.96**  
▸ **paid_search   :      922374.20**  
▸ **social        :    976818.37**  
  

In [13]:
#-----------------------------------------------------------------PARTE 1----------------------------------------------------------
print("-------------Ingreso_total---------------")
Ingreso_total = orders["monto_total"].sum()
print(f"Ingreso_total:${Ingreso_total}")

-------------Ingreso_total---------------
Ingreso_total:$52002629.56


In [14]:
print("-------------Costo_total---------------")
# Unir orders con catalog usando nombre_producto
df_costo = orders.merge(catalog, on='nombre_producto', how='left')

# Columna nueva con costo total
df_costo['costo_total'] = df_costo['cantidad'] * df_costo['costo_unitario']

# Sumar todo para obtener el costo total
costo_total = round(df_costo['costo_total'].sum(),2)

print(f"Costo total: ${costo_total}")

-------------Costo_total---------------
Costo total: $43140620.46


In [15]:
print("-------------Gasto por campaña---------------")
gasto_por_campaña = marketing.groupby('id_campaña')['gasto'].sum()
print(gasto_por_campaña)
print("-------------Gasto total de Marketing---------------")
gasto_total = marketing["gasto"].sum()
print(gasto_total)

-------------Gasto por campaña---------------
id_campaña
organic_Argentina        319196.99
organic_Colombia         323222.51
organic_Mexico           330231.46
paid_search_Argentina    297321.98
paid_search_Colombia     298901.96
paid_search_Mexico       326150.26
social_Argentina         331175.63
social_Colombia          313528.95
social_Mexico            332113.79
Name: gasto, dtype: float64
-------------Gasto total de Marketing---------------
2871843.53


In [16]:
print("-----------Profit----------------")
profit_neto = round(Ingreso_total - costo_total - gasto_total,2)
print(f"Profit neto: ${profit_neto}")

-----------Profit----------------
Profit neto: $5990165.57


In [17]:
#-----------------------------------------------------------------PARTE 1-----------------------------------------
print("---------------------------Ticket Promedio-------------------------------------")
numero_pedidos = orders['id_pedido'].nunique()
ticket_promedio = round(Ingreso_total / numero_pedidos, 2)
print(f"Ticket promedio por pedido: ${ticket_promedio:,.2f}")
print()
print("------------------------Cantidad Promedio de Productos por Orden---------------")
cantidad_total = orders['cantidad'].sum()
numero_pedidos = orders['id_pedido'].nunique()
cantidad_promedio = round(cantidad_total / numero_pedidos, 2)
print(f"Cantidad promedio de productos por orden: {cantidad_promedio}u")
print()
print("----------------------------Producto Más Vendido-------------------------------")
ventas_por_producto = df_costo.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False)
producto_mas_vendido = ventas_por_producto.idxmax()
cantidad_vendida = ventas_por_producto.max()
print(f"El producto más vendido es: {producto_mas_vendido} con {cantidad_vendida} unidades vendidas")
print()
print("-------------------------------Gasto por canal---------------------------------")
gasto_por_campaña = marketing.groupby('canal')['gasto'].sum()
print(gasto_por_campaña)
print()
print("--------------------------------Gasto total de Marketing-----------------------")
gasto_total = marketing["gasto"].sum()
print(gasto_total)

---------------------------Ticket Promedio-------------------------------------
Ticket promedio por pedido: $2,084.27

------------------------Cantidad Promedio de Productos por Orden---------------
Cantidad promedio de productos por orden: 7.12u

----------------------------Producto Más Vendido-------------------------------
El producto más vendido es: Laptop-Gaming-16GB con 144219.0 unidades vendidas

-------------------------------Gasto por canal---------------------------------
canal
organic        972650.96
paid_search    922374.20
social         976818.37
Name: gasto, dtype: float64

--------------------------------Gasto total de Marketing-----------------------
2871843.53


## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario

▸ first_visit	**7796**  

▸ add_to_cart	**7634**  

▸ add_payment_info	**6250**  

▸ purchase	**6240**  

▸ begin_checkout	**7208**  

▸ select_item	**7582**  


---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [18]:

import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})


In [19]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head(3)

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica


In [20]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT  nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios
FROM events
GROUP BY nombre_evento
ORDER BY 
    CASE nombre_evento
        WHEN 'first_visit' THEN 1
        WHEN 'add_to_cart' THEN 2
        WHEN 'add_payment_info' THEN 3
        WHEN 'purchase' THEN 4
    END;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios
0,first_visit,7796
1,add_to_cart,7634
2,add_payment_info,6250
3,purchase,6240
4,begin_checkout,7208
5,select_item,7582


In [21]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH funnel AS (
    SELECT 
        COUNT(DISTINCT CASE WHEN nombre_evento = 'first_visit' THEN id_usuario END) AS first_visit,                                        --conteo unicos de paso 1
        COUNT(DISTINCT CASE WHEN nombre_evento = 'add_to_cart' THEN id_usuario END) AS add_to_cart,                                        --conteo unicos de paso 2
        COUNT(DISTINCT CASE WHEN nombre_evento = 'add_payment_info' THEN id_usuario END) AS add_payment_info,                              --conteo unicos de paso 3
        COUNT(DISTINCT CASE WHEN nombre_evento = 'purchase' THEN id_usuario END) AS purchase                                               --conteo unicos de paso 4
    FROM events                                                                                                                            --la fuente
)
SELECT                 ---------------------------------------------Conteo unico para cada paso---------------------------------------------------
    first_visit,
    add_to_cart,
    add_payment_info,
    purchase,

                    ---------------------------------------- Tasa de conversión entre cada paso--------------------------------------------------------------
    ROUND(add_to_cart * 100.0 / first_visit, 2) AS conv_visita_a_carrito,
    ROUND(add_payment_info * 100.0 / add_to_cart, 2) AS conv_carrito_a_pago,
    ROUND(purchase * 100.0 / add_payment_info, 2) AS conv_pago_a_compra,

                     ------------------Tasa de abandono entre cada paso (para identificar dónde se pierden más usuarios)------------------------------------
    ROUND((first_visit - add_to_cart) * 100.0 / first_visit, 2) AS drop_visita_a_carrito,
    ROUND((add_to_cart - add_payment_info) * 100.0 / add_to_cart, 2) AS drop_carrito_a_pago,
    ROUND((add_payment_info - purchase) * 100.0 / add_payment_info, 2) AS drop_pago_a_compra,

                    ---------------------------------------Tasa de conversión final (del primer paso al último)------------------------------------------------
    ROUND(purchase * 100.0 / first_visit, 2) AS conversion_final

FROM funnel;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,first_visit,add_to_cart,add_payment_info,purchase,conv_visita_a_carrito,conv_carrito_a_pago,conv_pago_a_compra,drop_visita_a_carrito,drop_carrito_a_pago,drop_pago_a_compra,conversion_final
0,7796,7634,6250,6240,97.92,81.87,99.84,2.08,18.13,0.16,80.04


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [22]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [23]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [24]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
--PASO 1: Identificar la cohorte de cada usuario según el mes de registro 
WITH cohortes AS (
    SELECT
        us.id_usuario,
        DATE_TRUNC('month', MIN(us.fecha_registro)::TIMESTAMP) AS cohorte_mes,
        ua.dias_despues_registro,
        ua.activo
    FROM users AS us
    JOIN user_activity AS ua ON us.id_usuario = ua.id_usuario
    GROUP BY us.id_usuario, ua.dias_despues_registro, ua.activo
),

--PASO 2: Calcular retención semanal por cohorte 
retencion AS (
    SELECT
        cohorte_mes,
        COUNT(DISTINCT id_usuario) AS clientes_iniciales,
        COUNT(DISTINCT CASE WHEN dias_despues_registro >= 7  AND activo = 1 THEN id_usuario END) AS retenido_w1,
        COUNT(DISTINCT CASE WHEN dias_despues_registro >= 14 AND activo = 1 THEN id_usuario END) AS retenido_w2,
        COUNT(DISTINCT CASE WHEN dias_despues_registro >= 28 AND activo = 1 THEN id_usuario END) AS retenido_w3
    FROM cohortes
    GROUP BY cohorte_mes
),
porcentaje_retencion AS (
    SELECT
        cohorte_mes,
        clientes_iniciales,
        retenido_w1,
        ROUND(100.0 * retenido_w1 / clientes_iniciales, 2) AS pct_retenido_w1,
        retenido_w2,
        ROUND(100.0 * retenido_w2 / clientes_iniciales, 2) AS pct_retenido_w2,
        retenido_w3,
        ROUND(100.0 * retenido_w3 / clientes_iniciales, 2) AS pct_retenido_w3
    FROM retencion
)

SELECT * FROM porcentaje_retencion
ORDER BY cohorte_mes;
                 
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte_mes,clientes_iniciales,retenido_w1,pct_retenido_w1,retenido_w2,pct_retenido_w2,retenido_w3,pct_retenido_w3
0,2025-01-01,1627,1381,84.88,1253,77.01,671,41.24
1,2025-02-01,1444,1255,86.91,1154,79.92,575,39.82
2,2025-03-01,1636,1428,87.29,1306,79.83,673,41.14
3,2025-04-01,1606,1394,86.80,1261,78.52,652,40.60
4,2025-05-01,1687,1446,85.71,1321,78.30,679,40.25


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística

   - **H₀ (Hipótesis nula):** No hay diferencia en la tasa de conversión entre control y tratamiento. Es decir, el cambio en la UI del checkout no tiene efecto sobre la conversión.
   - **H₁ (Hipótesis alternativa):** Hay diferencia en la tasa de conversión entre control y tratamiento. Es decir, el cambio en la UI del checkout tiene efecto sobre la conversión.

   
**Test estadístico:** Prueba Z para binario (Convirtio) y categorico (Variante)
**Nivel de significancia alpha:** 0.05 y obtuvimos un p_valor de 0.41

In [25]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest
df = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')

In [26]:
#df.info()
#------------------------------------Prueba Z para binario y categorico-----------------
control = df[df.variante == "control"]
tratamiento = df[df.variante == "tratamiento"]
Exitos = [control.convirtio.sum(), tratamiento.convirtio.sum()]
Observaciones = [len(control), len(tratamiento)]
z_stat, p_value = proportions_ztest(Exitos, Observaciones)
print(f"Estadistico_Z: {z_stat:.4f}, p-value: {p_value:.4f}")

Estadistico_Z: -0.8133, p-value: 0.4161


In [27]:
#--------------------------------------Automatización de hipotesis----------------------
alpha = 0.05
if p_value < alpha:
    print("Rechazamos la Hipótesis nula: hay evidencia estadística de diferencia")
else:
    print("No rechazamos la Hipótesis nula: no hay evidencia estadística de diferencia")
print()
p_control = Exitos[0]/Observaciones[0]
p_trat = Exitos[1]/Observaciones[1]
print(f"Tasa control: {p_control:.4f}, Tasa tratamiento: {p_trat:.4f}")
print(f"Lift: {(p_trat - p_control)*100:.2f} puntos porcentuales")

No rechazamos la Hipótesis nula: no hay evidencia estadística de diferencia

Tasa control: 0.1569, Tasa tratamiento: 0.1629
Lift: 0.60 puntos porcentuales


- **las tasas de conversion son ligeramente diferentes observado por un valor z de valor cercano a 0**
- **El P_valor es mayor que alpha por lo que no hay evidencia estadistica de diferencia entre las tasa de conversion**
- **Conclusión: No hay evidencia de diferencia real entre las tasa de conversion, no hubo impacto en UI CHECKOUT**
  

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [ ]:
# (Pega aquí tu link)
# link de power bi o tableau
# link de one drive / google drive
https://drive.google.com/drive/folders/1vxtHg6imzAH-JHhD2gf0qfDM3AXmnpeo?usp=sharing